In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout

from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = pd.read_csv("../data/cleaned/spam.csv")

df = df.dropna(subset=['clean_message'])
df = df[df['clean_message'].str.strip() != ""]

In [3]:
encoder = LabelEncoder()

df["label"] = encoder.fit_transform(df["label"])

In [4]:
tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(df["clean_message"])

X = tokenizer.texts_to_sequences(df["clean_message"])

In [ ]:
# Ensure tokenizer exists and is fitted, then create padded sequences
try:
	tokenizer
except NameError:
	tokenizer = Tokenizer(num_words=5000)

if not hasattr(tokenizer, "word_index") or len(tokenizer.word_index) == 0:
	tokenizer.fit_on_texts(df["clean_message"])

sequences = tokenizer.texts_to_sequences(df["clean_message"])
X = pad_sequences(sequences, maxlen=100)

ValueError: `sequences` must be iterable.

In [5]:
X = pad_sequences(X, maxlen=100)

y = df["label"]

In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df["clean_message"])

In [7]:
import pickle

with open("../models/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Tokenizer saved successfully!")

Tokenizer saved successfully!


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [9]:
gru_model = Sequential()

gru_model.add(Embedding(input_dim=5000, output_dim=64))

gru_model.add(GRU(64))

gru_model.add(Dropout(0.5))

gru_model.add(Dense(1, activation="sigmoid"))

gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
history = gru_model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.9115 - loss: 0.2553 - val_accuracy: 0.9764 - val_loss: 0.0957
Epoch 2/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.9860 - loss: 0.0503 - val_accuracy: 0.9809 - val_loss: 0.0765
Epoch 3/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.9941 - loss: 0.0218 - val_accuracy: 0.9798 - val_loss: 0.0800
Epoch 4/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.9983 - loss: 0.0112 - val_accuracy: 0.9776 - val_loss: 0.0957
Epoch 5/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.9986 - loss: 0.0062 - val_accuracy: 0.9787 - val_loss: 0.0912


In [11]:
loss, accuracy = gru_model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9856 - loss: 0.0526
Test Accuracy: 0.985637366771698


In [12]:
y_pred = (gru_model.predict(X_test) > 0.5).astype(int)

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step


In [13]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       979
           1       0.95      0.93      0.94       135

    accuracy                           0.99      1114
   macro avg       0.97      0.96      0.97      1114
weighted avg       0.99      0.99      0.99      1114



In [15]:
gru_model.save("../models/gru_model.keras")